In [66]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("C:\\Users\\lenevo\\Downloads\\loan_approval_dataset.csv")

df.columns = df.columns.str.strip() # strips the whitespace in the column names

# print(df.head())
# print(df.info())
# print(df.describe())
print(df.columns.tolist())

# print(df.isnull()) # No missing values

df_encoded = pd.get_dummies(df, columns=['education', 'self_employed'], drop_first=True)
# print(df_encoded)

df_encoded['loan_status'] = df_encoded['loan_status'].str.strip()
df_encoded['loan_status'] = df_encoded['loan_status'].map({'Approved': 1, 'Rejected': 0})

X = df_encoded.drop('loan_status', axis=1)
y = df_encoded['loan_status']

# print(X)
# print(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale")
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, pos_label=1)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    confusion = confusion_matrix(y_test, predictions)

    results.append({
        "Model": name,
        "Accuracy Score": accuracy,
        "Precision Score": precision,
        "Recall Score": recall,
        "f1 Score": f1,
        "Confusion Matrix": confusion
    })

# print(results)

results_df = pd.DataFrame(results)
print()
print(results_df)

# The best model is Random Forest

model = RandomForestClassifier(random_state=42)

params = {              # Parameters for Random Forest only
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10]
}

search = RandomizedSearchCV(
    model,
    params,
    n_iter=10,
    cv = 5,
    scoring = "accuracy",
    random_state = 42
)

search.fit(X_train, y_train)
print()
print(search.best_params_)

best_model = search.best_estimator_

predictions = best_model.predict(X_test)

print(accuracy_score(y_test, predictions))

['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']

                 Model  Accuracy Score  Precision Score  Recall Score  \
0  Logistic Regression        0.907494         0.920810      0.932836   
1        Random Forest        0.978923         0.983209      0.983209   
2                  SVM        0.935597         0.959847      0.936567   

   f1 Score        Confusion Matrix  
0  0.926784  [[275, 43], [36, 500]]  
1  0.983209    [[309, 9], [9, 527]]  
2  0.948064  [[297, 21], [34, 502]]  

{'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 20}
0.9730679156908665
